## CALL DETAILED REPORT

In [1]:
import pandas as pd
import numpy as np
import math
import gspread
from gspread_pandas import Spread, conf
from datetime import datetime, timedelta

In [2]:
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
c = conf.get_config(file_name=config_path)
report_sheet_url = "https://docs.google.com/spreadsheets/d/1PqMNtFU0bas_BGYFhIYWojO2petW-TOGxeRB2OWnF08/edit?pli=1&gid=772676767#gid=772676767" 
spread = Spread(report_sheet_url, config=c)

In [3]:
target_sheets = [s.title for s in spread.sheets if s.title.lower().startswith('3cx')]

try:
    existing_df = spread.sheet_to_df(sheet='Call_Report_Summary', index=None)
    if not existing_df.empty and 'Date' in existing_df.columns:
        existing_dates = set(pd.to_datetime(existing_df['Date']).dt.strftime('%Y-%m-%d').tolist())
    else:
        existing_dates = set()
except Exception:
    existing_df = pd.DataFrame()
    existing_dates = set()

all_new_data = []


for sheet_name in target_sheets:
    try:
        date_str = sheet_name.split('_')[-1]
        date_obj = pd.to_datetime(date_str, format='%d-%m-%y')
        formatted_date = date_obj.strftime('%Y-%m-%d')
        
        if formatted_date in existing_dates:
            print(f"Skipping {sheet_name}: Date {formatted_date} already in summary.")
            continue

        df = spread.sheet_to_df(index=None, sheet=sheet_name, start_row=3)
        
        if df.empty:
            continue

        # Handle duplicate columns in the source sheet
        if df.columns.duplicated().any():
            df = df.loc[:, ~df.columns.duplicated()]
        
        # Calculate Week of the Month
        week_of_month = math.ceil(date_obj.day / 7)
        
        # Insert Date and Week info
        df.insert(0, 'Date', formatted_date)
        df.insert(1, 'Week', f"Week {week_of_month}")
        
        all_new_data.append(df)
        print(f"Processed new data from: {sheet_name}")

    except Exception as e:
        print(f"Error processing {sheet_name}: {e}")


if all_new_data:
    new_data_combined = pd.concat(all_new_data, ignore_index=True)
    
    master_df = pd.concat([existing_df, new_data_combined], ignore_index=True)
    
    if 'BDE' in master_df.columns:
        # Remove empty rows or header-repeat rows in BDE column
        master_df = master_df[master_df['BDE'].astype(str).str.strip() != '']
        master_df = master_df[master_df['BDE'] != 'BDE']
        master_df = master_df.dropna(subset=['BDE'])

    # Ensure numeric columns are strictly numbers
    numeric_cols = ['Dailed Count', 'Answered', 'Unanswered', 'Total Duration (minutes)', 'Avg Minutes per Answered Call']
    
    for col in numeric_cols:
        if col in master_df.columns:
            master_df[col] = pd.to_numeric(master_df[col], errors='coerce').fillna(0)

    # 5. Final Sort: Date (Ascending) and BDE (Ascending)
    master_df = master_df.sort_values(by=['Date', 'BDE'], ascending=[True, True])

    # Standardize column order
    cols_order = ['Date', 'Week', 'BDE', 'Dailed Count', 'Answered', 'Unanswered', 'Total Duration (minutes)', 'Avg Minutes per Answered Call']
    
    existing_cols = [c for c in cols_order if c in master_df.columns]
    master_df = master_df[existing_cols]

    # Write back to 'Call_Report_Summary'
    spread.df_to_sheet(master_df, index=False, sheet='Call_Report_Summary', replace=True)
    print("Update successful! New records appended and sorted.")
else:
    print("No new dates to add. Summary is already up to date.")

Skipping 3cx_report_06-03-26: Date 2026-03-06 already in summary.
Skipping 3cx_report_07-03-26: Date 2026-03-07 already in summary.
Processed new data from: 3cx_report_09-03-26
Processed new data from: 3cx_report_10-03-26
Processed new data from: 3cx_report_11-03-26
Update successful! New records appended and sorted.


In [4]:
master_df.to_excel("./Output_Data/call_report_summary.xlsx")